# pyrepl

> A Python prompt you own, and an agent in the layer above it.

One terminal holds two things. The Python prompt's namespace belongs to whoever is typing;
the agent reads that namespace and builds in a layer of its own, and cannot rebind a name it
did not create. That guarantee is not this module's: Dhrishti serves the kernel's namespace
and splits its API in two, and everything here does is point the agent at the half that
cannot mutate anything.

In [ ]:
#| default_exp pyrepl

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import test_eq, test_fail

In [ ]:
#| export
import asyncio, codeop, json, os, queue, re, shutil, sys, tempfile, threading, urllib.parse, urllib.request
from dataclasses import dataclass, field
from pathlib import Path
from rich.text import Text
from ramabana.core import agent_err
from ramabana.tools import LocalHost, WRITE_TOOLS
from ramabana.agent import Agent, Approvals
from ramabana.cli import Ui, GRUVBOX, HELP, attach_refs, media_parts, media_note

## Outputs

A kernel reports its results as a stream of messages; a notebook stores them as a list of
dicts. `ExecOutcome` is one request in the notebook's shape, which is what lets the same
outputs go to the terminal, to `log_cell` and to a test without a second representation.

In [ ]:
#| export
@dataclass
class ExecOutcome:
    "One kernel request in nbformat's output shape."
    ok: bool = True
    outputs: list = field(default_factory=list)
    execution_count: int | None = None
    error: str | None = None

def output_text(outputs):
    "Flatten notebook outputs for tests, logs and plain terminal fallbacks."
    parts = []
    for out in outputs:
        kind = out.get('output_type')
        if kind == 'stream': parts.append(_text(out.get('text')))
        elif kind in ('execute_result', 'display_data'):
            data = out.get('data') or {}
            parts.append(_text(data.get('text/plain') or data.get('text/markdown')))
        elif kind == 'error':
            trace = out.get('traceback') or []
            parts.append('\n'.join(trace) if trace else f"{out.get('ename')}: {out.get('evalue')}")
    return '\n'.join(p.rstrip('\n') for p in parts if p is not None)

def _text(value):
    return ''.join(value) if isinstance(value, list) else str(value or '')

In [ ]:
outs = [{'output_type': 'stream', 'text': 'hello\n'},
        {'output_type': 'execute_result', 'data': {'text/plain': '42'}},
        {'output_type': 'error', 'ename': 'ValueError', 'evalue': 'bad', 'traceback': []}]
test_eq(output_text(outs), 'hello\n42\nValueError: bad')
# A stream arrives split across messages, so the list form has to flatten rather than repr.
test_eq(output_text([{'output_type': 'stream', 'text': ['a', 'b']}]), 'ab')
# An error with a traceback prefers it: the ename alone loses where it happened.
test_eq(output_text([{'output_type': 'error', 'ename': 'E', 'evalue': 'v',
                      'traceback': ['line one', 'line two']}]), 'line one\nline two')
test_eq(output_text([]), '')

## The kernel

A private `ipykernel`, and Dhrishti started *inside* it — because the namespace Dhrishti
serves is the kernel's, and because its server runs on a background thread, which is what
keeps it answering during exactly the long cell you most want to watch.

The port comes back through a printed marker rather than a return value: the bootstrap runs
as a cell, and a cell's only channel to the caller is its output.

In [ ]:
#| export
class Kernel:
    "A private ipykernel with Dhrishti serving its live namespace."
    def __init__(self, cwd='.'):
        self.cwd = Path(cwd).resolve()
        self.km = self.kc = None
        self.base = None
        self._ipc_dir = None
        self._exec_lock = asyncio.Lock()
        self._shell_lock = asyncio.Lock()

    @property
    def alive(self):
        return self.km is not None and self.kc is not None and self.km.has_kernel

    async def start(self, timeout=60):
        from jupyter_client.kernelspec import KernelSpec
        from jupyter_client.manager import AsyncKernelManager
        opts = {'kernel_name': 'python3'}
        if os.name != 'nt':
            self._ipc_dir = tempfile.mkdtemp(prefix='rama-k-', dir='/tmp' if os.path.isdir('/tmp') else None)
            opts.update(transport='ipc', ip=os.path.join(self._ipc_dir, 'k'))
        self.km = AsyncKernelManager(**opts)
        self.km._kernel_spec = KernelSpec(
            argv=[sys.executable, '-m', 'ipykernel_launcher', '-f', '{connection_file}'],
            display_name='Ramabana PyREPL', language='python')
        try:
            await self.km.start_kernel(cwd=str(self.cwd))
            self.kc = self.km.client()
            self.kc.start_channels()
            await self.kc.wait_for_ready(timeout=timeout)
            await self._bootstrap()
            return self
        except BaseException:
            await self.shutdown()
            raise

    async def _bootstrap(self):
        root = self.cwd/'.ramabana'/'pyrepl'
        # Named '<project>-pyrepl-<pid>', not a bare "ramabana pyrepl": every kernel used to
        # register under that same literal string, so two live sessions (this project run
        # twice, or a stale process from an earlier run) were indistinguishable in the
        # registry and a by-name attach could silently resolve to the wrong one. The pid makes
        # it unique; `find_session`'s project-prefix match is what keeps a human from having to
        # type the pid to use it.
        #
        # The basename is sanitised, not taken verbatim: a name is a UI a human types at
        # --attach, so 'My Project' becomes 'my-project' rather than something that needs
        # careful quoting. `self.cwd.name` is only empty when cwd resolves to a filesystem
        # root (a container rooted at '/') -- rare, and the pid still keeps the full name
        # unique there, so every such session sharing the 'ramabana' fallback segment just
        # means prefix matching degrades to needing the exact name in that one case. Left as
        # is rather than handled further.
        proj = re.sub(r'[^a-z0-9]+', '-', (self.cwd.name or 'ramabana').lower()).strip('-') or 'ramabana'
        source = (
            'def _ramabana_bootstrap():\n'
            ' import dhrishti.serving as ds, os\n'
            f' p = ds.serve_in_kernel(name={proj!r} + "-pyrepl-" + str(os.getpid()), agent="restricted", token=True, '
            f'session_dir={str(root/"sessions")!r}, agent_session_dir={str(root/"agents")!r})\n'
            ' print("__RAMABANA_DHRISHTI__" + str(p))\n'
            '_ramabana_bootstrap()\n'
            'del _ramabana_bootstrap')
        result = await self.execute(source, store_history=False)
        marker = next((line for out in result.outputs if out.get('output_type') == 'stream'
                       for line in _text(out.get('text')).splitlines()
                       if line.startswith('__RAMABANA_DHRISHTI__')), '')
        if not result.ok or not marker:
            raise RuntimeError(result.error or output_text(result.outputs) or 'Dhrishti did not start')
        self.base = f'http://127.0.0.1:{int(marker.removeprefix("__RAMABANA_DHRISHTI__"))}'

    async def execute(self, code, store_history=True, on_output=None):
        "Execute one cell and stream each nbformat-shaped output to `on_output`."
        if not self.alive: return ExecOutcome(ok=False, error='kernel is not running')
        async with self._exec_lock, self._shell_lock:
            msg_id = self.kc.execute(str(code), store_history=store_history, allow_stdin=False)
            outputs = await self._collect(msg_id, on_output)
            reply = await self._reply(msg_id)
        content = (reply or {}).get('content') or {}
        error = None
        if content.get('status') == 'error': error = f"{content.get('ename')}: {content.get('evalue')}"
        elif content.get('status') == 'abort': error = 'aborted'
        if error is None:
            err = next((o for o in outputs if o.get('output_type') == 'error'), None)
            if err: error = f"{err.get('ename')}: {err.get('evalue')}"
        return ExecOutcome(error is None, outputs, content.get('execution_count'), error)

    async def _collect(self, msg_id, on_output):
        outputs, displays = [], {}
        while True:
            try: message = await self.kc.get_iopub_msg(timeout=.5)
            except (queue.Empty, asyncio.TimeoutError):
                if not self.alive: break
                continue
            if (message.get('parent_header') or {}).get('msg_id') != msg_id: continue
            kind, content = message['header']['msg_type'], message['content']
            if kind == 'status' and content.get('execution_state') == 'idle': break
            if kind == 'clear_output': outputs.clear(); displays.clear(); continue
            out = self._output(kind, content)
            if out is None: continue
            display_id = (content.get('transient') or {}).get('display_id')
            if kind == 'update_display_data' and display_id in displays:
                outputs[displays[display_id]] = out
            elif (out['output_type'] == 'stream' and outputs and outputs[-1].get('output_type') == 'stream'
                  and outputs[-1].get('name') == out.get('name')):
                outputs[-1]['text'] += out['text']
            else:
                outputs.append(out)
                if display_id: displays[display_id] = len(outputs) - 1
            if on_output: on_output(out)
        return outputs

    @staticmethod
    def _output(kind, content):
        if kind == 'stream':
            return {'output_type': 'stream', 'name': content.get('name', 'stdout'), 'text': content.get('text', '')}
        if kind == 'error':
            return {'output_type': 'error', 'ename': content.get('ename', ''),
                    'evalue': content.get('evalue', ''), 'traceback': list(content.get('traceback') or [])}
        if kind in ('execute_result', 'display_data', 'update_display_data'):
            return {'output_type': 'execute_result' if kind == 'execute_result' else 'display_data',
                    'data': content.get('data') or {}, 'metadata': content.get('metadata') or {}}
        return None

    async def _reply(self, msg_id):
        while True:
            try: message = await self.kc.get_shell_msg(timeout=10)
            except (queue.Empty, asyncio.TimeoutError): return None
            if (message.get('parent_header') or {}).get('msg_id') == msg_id: return message

    async def complete(self, code, pos):
        async with self._shell_lock:
            msg_id = self.kc.complete(str(code), int(pos))
            while True:
                message = await self.kc.get_shell_msg(timeout=10)
                if (message.get('parent_header') or {}).get('msg_id') == msg_id:
                    content = message.get('content') or {}
                    return content.get('matches') or [], int(content.get('cursor_start', pos))

    async def interrupt(self):
        if self.km: await self.km.interrupt_kernel()

    async def shutdown(self):
        if self.kc:
            try: self.kc.stop_channels()
            except Exception: pass
        if self.km and self.km.has_kernel:
            try: await self.km.shutdown_kernel(now=False)
            except Exception: pass
        self.km = self.kc = self.base = None
        if self._ipc_dir:
            shutil.rmtree(self._ipc_dir, ignore_errors=True)
            self._ipc_dir = None

In [ ]:
kernel = await Kernel('.').start()
r = await kernel.execute('a = 6 * 7\na')
test_eq((r.ok, output_text(r.outputs)), (True, '42'))
assert kernel.base.startswith('http://127.0.0.1:')     # dhrishti came up inside the kernel

In [ ]:
# A dead kernel answers rather than hanging on a channel nobody is writing to.
dead = Kernel('.')
r = await dead.execute('1 + 1')
test_eq((r.ok, r.error), (False, 'kernel is not running'))

# An error is an outcome, not an exception: the caller is a UI.
r = await kernel.execute('1/0')
test_eq(r.ok, False)
assert 'ZeroDivisionError' in r.error and 'ZeroDivisionError' in output_text(r.outputs)

# Streams are coalesced, so a loop printing forty lines is one output.
r = await kernel.execute("for i in range(3): print(i)")
test_eq(len([o for o in r.outputs if o['output_type'] == 'stream']), 1)
test_eq(output_text(r.outputs), '0\n1\n2')

# Completion comes from the kernel, which is IPython's completer with jedi behind it.
await kernel.execute('import json')
matches, start = await kernel.complete('json.du', 7)
assert 'json.dump' in matches or 'dump' in matches

## The host

An ordinary Ramabana host — the file, search and web tools are the same ones — whose *Python*
tools go over HTTP to the agent half of the Dhrishti API. That half is the ungated one, and
that is the whole protection: the token that opens `/api/exec`, `/api/set` and `/api/promote`
is never read here, so there is no code path from a tool call to the owner's namespace.

`log_cell` is what makes the session one artifact. The owner's cells go in as code with their
real outputs, each turn as `**user**`/`**assistant**` markdown, in the order they happened.

In [ ]:
#| export
def _api(base, path, params=None, timeout=60):
    query = urllib.parse.urlencode(params or {})
    url = base.rstrip('/') + path + (('?' + query) if query else '')
    with urllib.request.urlopen(url, timeout=timeout) as response:
        return json.loads(response.read())

class DhrishtiHost(LocalHost):
    "A project host whose Python tools use a protected Dhrishti overlay."
    def __init__(self, roots, base, **kwargs):
        super().__init__(roots, **kwargs)
        self.base = base
        # A dead or slow-to-start base must not raise here: this runs at construction, before
        # a caller has a host to retry against, so it gets the same agent_err treatment as
        # every other transport call below rather than an unhandled exception.
        try: info = _api(base, '/agent/api/info')
        except Exception: info = {}
        self.agent_log = Path((info or {}).get('log') or '')

    def run_python(self, code):
        # `scope='overlay'`, always. The agent gets the sandboxed layer whatever it asks for,
        # because a scope is a tool argument and a tool argument is a thing a model can set.
        try: result = _api(self.base, '/agent/api/exec', {'code': str(code), 'scope': 'overlay'})
        except Exception as exc: return agent_err(exc)
        if result.get('error'): return str(result['error'])
        out = result.get('stdout') or ''
        value = result.get('result')
        if isinstance(value, dict): value = value.get('value')
        if value is not None: out += ('\n' if out else '') + str(value)
        return out or '(ok)'

    def inspect_python(self, code, scope='isolated'):
        if scope not in self.scopes: return f'this host only honours {self.scopes}'
        try: result = _api(self.base, '/agent/api/exec', {'code': str(code), 'scope': scope})
        except Exception as exc: return agent_err(exc)
        if result.get('error'): return str(result['error'])
        out = result.get('stdout') or ''
        value = result.get('result')
        if isinstance(value, dict): value = value.get('value')
        if value is not None: out += ('\n' if out else '') + str(value)
        return out or '(ok)'

    @property
    def scopes(self): return ('isolated', 'overlay')

    @property
    def kernel_kind(self): return 'ipykernel'

    def list_vars(self):
        try: result = _api(self.base, '/agent/api/rows', {'profile': 'minimal', 'sort': 'name'})
        except Exception as exc: return agent_err(exc)
        return '\n'.join(f"{node.get('name')}: {node.get('type')} = {node.get('value')}"
                         for group in result.get('groups', []) for node in group.get('nodes', []))

    def log_cell(self, source, outputs=None, cell_type='code'):
        "Append a human Python or model markdown turn to Dhrishti's session notebook."
        # `read`/append/`write` per cell rather than a held handle: the owner's kernel writes
        # this file too, and a session that survives a crash is worth more than the syscalls.
        if not str(self.agent_log): return
        from fastcore.nbio import read_nb, write_nb, new_nb, mk_cell
        self.agent_log.parent.mkdir(parents=True, exist_ok=True)
        nb = read_nb(self.agent_log) if self.agent_log.exists() else new_nb([])
        cell = mk_cell(str(source), cell_type)
        if cell_type == 'code':
            cell['outputs'], cell['execution_count'] = list(outputs or []), None
        nb.cells.append(cell)
        write_nb(nb, self.agent_log)

    def describe(self):
        "name -> 'type [shape]' for everything the session is holding. Never raises."
        try: result = _api(self.base, '/agent/api/rows', {'profile': 'minimal', 'sort': 'name'}, timeout=2)
        except Exception: return {}
        out = {}
        for group in result.get('groups', []):
            for node in group.get('nodes', []):
                if name := node.get('name'):
                    # `is not None` rather than truthy: a present-but-falsy value (`0`, `''`, an
                    # empty container) is still the value, and only a genuinely missing key
                    # falls back to the type. In practice dhrishti's own `value_str` always
                    # reprs a live binding to a non-empty string, so this exact mislabelling
                    # cannot happen against the real API today -- but the check should say what
                    # it means rather than lean on that staying true forever.
                    value = node.get('value')
                    out[name] = str(value if value is not None else (node.get('type') or ''))
        return out

In [ ]:
host = DhrishtiHost(['.'], kernel.base, web=False, index=False)
await kernel.execute("owner = {'kept': 1}")

test_eq(host.run_python('mine = len(owner)'), '(ok)')      # reads the owner freely
assert 'mine' in host.list_vars() and 'owner' in host.list_vars()

# The layering, from this side of it: the write lands in the agent's layer and the owner's
# binding is untouched. Dhrishti refuses it; this only has to not ask for a way round.
host.run_python('owner = None')
r = await kernel.execute('owner')
test_eq(output_text(r.outputs), "{'kept': 1}")

# A transport that is not there comes back as a sentence, because a tool cannot raise usefully.
assert 'Error' in DhrishtiHost(['.'], 'http://127.0.0.1:1', web=False, index=False).run_python('1')

In [ ]:
from fastcore.nbio import read_nb
r = await kernel.execute('logged = 1')
host.log_cell('logged = 1', r.outputs)
host.log_cell('**user**\n\nwhat is logged?', cell_type='markdown')
nb = read_nb(host.agent_log)
test_eq([c.cell_type for c in nb.cells[-2:]], ['code', 'markdown'])
test_eq(nb.cells[-2].outputs, r.outputs)          # the real outputs, not a rendering of them

In [ ]:
# `describe`'s own fallback, pinned directly: a present-but-falsy value (`0`, `''`) is still
# the value, and only a genuinely absent key (`None`) falls back to the type. This is checked
# against a canned `_api` rather than a live `0` binding: dhrishti's `value_str` always reprs a
# real value to a non-empty string ('0', "''", ...), so a live `int` binding holding `0` never
# actually exercises the buggy branch this guards -- the raw-falsy case only arises from a
# transport that hands back JSON values rather than dhrishti's own repr strings, which is
# exactly what this fakes.
_real_api = _api
def _fake_rows_api(base, path, params=None, timeout=60):
    return {'groups': [{'nodes': [
        {'name': 'zero_val', 'type': 'int', 'value': 0},
        {'name': 'blank_val', 'type': 'str', 'value': ''},
        {'name': 'missing_val', 'type': 'NoneType', 'value': None},
    ]}]}
_api = _fake_rows_api
try: described = host.describe()
finally: _api = _real_api
test_eq(described['zero_val'], '0')          # falsy but present: the value, not the type
test_eq(described['blank_val'], '')          # same for an empty string
test_eq(described['missing_val'], 'NoneType')  # genuinely absent: falls back to the type

### Completion, with the namespace described

Two sources, and the split is the point. **The kernel completes** — `complete_request` is
IPython's completer with jedi behind it, so it knows imports, attribute chains, dict keys and
paths. Dhrishti has no completion endpoint at all; its API is rows, expand, grid, result,
history, sessions, and rows only knows top-level bindings, so a completer built on it would
be a downgrade.

**Dhrishti describes.** Each candidate that names something live gains its type and shape, so
the row reads `df → DataFrame [1200×8]` rather than `df`. Describing a namespace needs no
token, which is why it comes from the agent half of the API.

Best-effort, always: the list paints from the kernel's answer and gains descriptions if they
arrive. A slow or broken lookup leaves bare names, never an empty list.

In [ ]:
#| export
def annotate(matches, described):
    "Candidate names with their type and shape where the session knows one."
    # Exact names only. `df.copy` is a method on a binding rather than a binding, and labelling
    # it with the frame's shape would say something untrue about it.
    return [f'{m} → {described[m]}' if m in described else m for m in matches]

In [ ]:
test_eq(annotate(['df', 'dfs'], {'df': 'DataFrame [3×2]'}), ['df → DataFrame [3×2]', 'dfs'])
test_eq(annotate(['x'], {}), ['x'])
test_eq(annotate([], {'x': 'int'}), [])
# An attribute chain is not a binding, so it stays bare rather than being mislabelled.
test_eq(annotate(['df.copy'], {'df': 'DataFrame [3×2]'}), ['df.copy'])

## Running it

`mk_pyagent` is the assembly: a `DhrishtiHost` over the folders named on the command line, an
`Agent` over that, and the approval gate wired to both. `amain` is the tty loop and it is
short, because everything it could get wrong lives in `PyreplUi`.

In [ ]:
#| export
def mk_pyagent(roots, base, model=None, approve='ask', web=True, read_outside=False, cfg=None):
    "Build an agent whose Python tools target the Dhrishti session at `base`."
    approvals = None if approve == 'none' else Approvals(tools=WRITE_TOOLS, mode=approve)
    host = DhrishtiHost(roots, base, approvals=approvals, web=web, read_outside=read_outside)
    agent = Agent(host, model=model, approvals=approvals, cfg=cfg)
    return agent, host

## Locking into a session someone else owns

Inside leela the kernel already exists and already has an agent session, so pyrepl starts
nothing: it finds the running server, builds the host against it, and runs the ordinary
terminal with no Python mode. Leela owns the human's prompt; Ramabana is the agent beside it.

Discovery goes through Dhrishti's registry, and **not** by calling `serve_in_kernel` again.
`_start` is guarded by `if _server is None`, but the lines above that guard are not: `_profile`,
`_env` and `_agent_mode` are reassigned unconditionally, and passing `session_dir` or
`agent_session_dir` calls `set_logging` regardless. Re-serving inside leela's kernel would hand
back the right port while repointing leela's session logs and resetting its agent mode -- a
`readonly` session would come back `restricted`. The registry is read-only, so it cannot.

Promotion is not Ramabana's to perform here. The agent's work is visible in the shared session
notebook and over the same API leela is already watching, and the human promotes from the
surface that holds the token.

In [ ]:
#| export
def _ambiguous(attach, cands):
    "Error text for a name that matches more than one live session -- list them, do not pick."
    rows = '; '.join(f"{e.get('name')} (cwd={e.get('cwd')}, port={e.get('port')})" for e in cands)
    return f'{attach!r} matches more than one live dhrishti session, refusing to guess which: {rows}'

def find_session(attach):
    """The base URL of a live Dhrishti session named `attach`, or `attach` itself if it is a URL.

    Through the registry rather than by re-serving. `serve_in_kernel` would return the right
    port for a kernel that is already serving -- and on the way there it reassigns the profile,
    the environment name and the agent mode, and repoints the session log directories. Attaching
    to somebody's session must not quietly widen it from `readonly` to `restricted`.

    Every match is refused rather than guessed at each stage: attach exists for exactly the case
    where another session is already running, so resolving to the wrong live namespace -- a
    different cwd, a different agent mode, someone else's data -- is worse than raising.
    """
    attach = str(attach).strip()
    if '://' in attach: return attach
    from dhrishti.serving import active
    entries = active()
    exact = [e for e in entries if e.get('name') == attach]
    if len(exact) > 1: raise RuntimeError(_ambiguous(attach, exact))
    hit = exact[0] if exact else None
    # A kernel registers as '<project>-pyrepl-<pid>' (see Kernel._bootstrap) so two sessions in
    # the same project never collide on name, but that also makes the full name unwieldy to
    # type -- so a name that isn't an exact match is tried as a project prefix next, and only
    # resolves if it picks out exactly one.
    if hit is None and attach:
        # Anchored to the project boundary, not a raw string prefix: 'ramabana' must find
        # 'ramabana-pyrepl-1' but not also 'ramabana-extra-pyrepl-2' -- a session from an
        # unrelated project that merely starts with the same letters is not a match at all.
        prefixed = [e for e in entries if str(e.get('name') or '').startswith(attach + '-pyrepl-')]
        if len(prefixed) > 1: raise RuntimeError(_ambiguous(attach, prefixed))
        hit = prefixed[0] if prefixed else None
    if hit is None and attach:
        # Falling back to cwd is a courtesy for a session started under a name this attach value
        # does not match at all; matched on the whole basename, not a substring, so 'leela' does
        # not also match 'old-leela-backup'.
        by_cwd = [e for e in entries if Path(str(e.get('cwd') or '')).name == attach]
        if len(by_cwd) > 1: raise RuntimeError(_ambiguous(attach, by_cwd))
        hit = by_cwd[0] if by_cwd else None
    if hit is None: raise RuntimeError(
        f'no live dhrishti session matching {attach!r}; running: '
        f'{", ".join(str(e.get("name")) for e in entries) or "none"}')
    return hit.get('base') or f'http://127.0.0.1:{hit["port"]}'

In [ ]:
# Anything with a scheme is taken as given, so an explicit URL never needs a registry.
test_eq(find_session('http://127.0.0.1:9999'), 'http://127.0.0.1:9999')
test_fail(lambda: find_session('no-such-session-anywhere'), contains='no live dhrishti session')

`find_session`'s resolution logic is pinned with synthetic registry entries rather than the
machine's real one -- the real registry is shared with every other dhrishti session on this
box (other repos, other terminals, even a stale leftover process), so a test that depended on
its contents would be exactly the class of test-reads-the-machine bug this notebook has spent
several tasks fixing. `unittest.mock.patch` swaps `dhrishti.serving.active` for the duration of
each assertion; `find_session` still runs its real resolution code against it.

In [ ]:
from unittest.mock import patch

def _entry(name, port, cwd='/x/proj'):
    return {'name': name, 'port': port, 'pid': port, 'cwd': cwd, 'base': f'http://127.0.0.1:{port}', 'started': 0}

one = [_entry('proj-pyrepl-111', 9001, '/home/proj')]
dupe_name = [_entry('proj-pyrepl-111', 9001, '/home/a'), _entry('proj-pyrepl-111', 9002, '/home/b')]
by_cwd = [_entry('some-other-name-222', 9003, '/home/leela')]

with patch('dhrishti.serving.active', lambda: one):
    # An exact, unique name resolves straight to its base.
    test_eq(find_session('proj-pyrepl-111'), 'http://127.0.0.1:9001')
    # A project prefix resolves too, when it picks out exactly one -- the ergonomic half of
    # the pid-qualified name: a human types 'proj', not 'proj-pyrepl-111'.
    test_eq(find_session('proj'), 'http://127.0.0.1:9001')

with patch('dhrishti.serving.active', lambda: dupe_name):
    # Two live entries sharing a name (the exact collision found live on this machine, between
    # a stale leftover and a fresh run) are refused, not guessed -- and named, so a human can
    # tell them apart rather than being told just "ambiguous".
    try:
        find_session('proj-pyrepl-111')
        assert False, 'expected a refusal'
    except RuntimeError as e:
        msg = str(e)
        assert 'matches more than one' in msg
        assert '/home/a' in msg and '/home/b' in msg   # both candidates identifiable, not just counted
        assert '9001' in msg and '9002' in msg

cross_project = [_entry('ramabana-pyrepl-1', 9004, '/home/ramabana'),
                 _entry('ramabana-extra-pyrepl-2', 9005, '/home/ramabana-extra')]

with patch('dhrishti.serving.active', lambda: cross_project):
    # The prefix match is anchored to the project boundary ('-pyrepl-'), not a raw string
    # prefix: 'ramabana' must resolve to exactly the one session that IS project 'ramabana',
    # not refuse just because 'ramabana-extra' happens to start with the same letters.
    test_eq(find_session('ramabana'), 'http://127.0.0.1:9004')

with patch('dhrishti.serving.active', lambda: by_cwd):
    # No name matches at all, so the whole-basename cwd fallback is what finds it.
    test_eq(find_session('leela'), 'http://127.0.0.1:9003')
    # A substring is not enough -- the footgun the fallback was tightened against.
    test_fail(lambda: find_session('eela'), contains='no live dhrishti session')

# A scheme is taken as given and never touches the registry at all.
test_eq(find_session('http://127.0.0.1:9999'), 'http://127.0.0.1:9999')

In [ ]:
#| export
async def amain(roots=('.',), model=None, approve='ask', web=True, read_outside=False,
                 cfg=None, resume='', attach=''):
    "Run the combined Ramabana Python and agent session, or attach to a session someone else owns."
    from teleprint.compositor import Compositor
    from teleprint.tty import RealTty
    # Attaching starts nothing and owns nothing: no kernel, no server, no token. The human's
    # Python prompt belongs to whoever started the session; Ramabana is only the agent beside it.
    kernel = None if attach else await Kernel(roots[0]).start()
    base = find_session(attach) if attach else kernel.base
    agent, host = mk_pyagent(roots, base, model, approve, web, read_outside, cfg)
    if resume: agent.resume_session(resume)
    # Bracketed paste on, mouse reporting off -- the same choice `cli.amain` makes, and for the
    # same reason: the main screen belongs to the terminal, so selecting and copying there work
    # as they do in any other scrollback. `Ui.enter_transcript` borrows the mouse for the
    # browsing view and gives it straight back. Holding it for the whole session both stole
    # that selection and streamed every mouse move in as input, which redrew the transcript
    # over and over -- the banner printed once per event.
    tty = RealTty(); tty.write('\x1b[?2004h')
    done = asyncio.Event()
    try:
        comp = await Compositor(tty).start()
        ui = PyreplUi(comp, agent, kernel, asyncio.get_running_loop()) if kernel else Ui(comp, agent, asyncio.get_running_loop())
        if kernel:
            ui.hint = f"{', '.join(host.roots)} · /agent asks · /python executes · /help"
        else:
            ui.hint = f"{', '.join(host.roots)} · attached to {base} · /help"
        comp.on_task_error = lambda exc, task: ui.say(Text(f'{task.get_name()} failed: {exc!r}'), 'error')
        comp.spawn(ui.animate(), name='spinner')
        def on_key(key):
            out = ui.on_key(key)
            if out == 'quit': return done.set()
            if out is not None:
                ui.turn = comp.spawn(out, name='pyrepl')
                ui.paint()
        comp.on_key, comp.on_paste = on_key, ui.paste
        comp.on_resize = lambda: (comp.resize(), ui.paint())
        comp.on_mouse = ui.transcript.on_mouse
        if kernel:
            subtitle = "Python owns the kernel; agent Python uses Dhrishti's protected overlay."
        else:
            subtitle = f"Attached to {base}; the human's Python prompt belongs to whoever started it."
        ui.say(Text('RAMABANA PYREPL', style=f"bold {GRUVBOX['fg0']}") +
               Text(f"\n\n{subtitle}", style=GRUVBOX['gray']),
               'note', fold=None)
        ui.paint()
        loop = asyncio.get_running_loop(); loop.add_reader(tty.fd, lambda: comp.on_bytes(tty.read(timeout=0)))
        try:
            while not done.is_set():
                try: await asyncio.wait_for(done.wait(), .2)
                except asyncio.TimeoutError: comp.flush_input()
        finally:
            loop.remove_reader(tty.fd); comp.stop()
    finally:
        tty.write('\x1b[?2004l\x1b[?1000;1006l\r\n'); tty.restore()
        agent.close()
        if kernel: await kernel.shutdown()

In [ ]:
#| export
def main(root:str='.',                     # folders the agent may touch, comma separated
         model:str=None,                   # the turn model; the routing default when omitted
         approve:str='ask',                # ask | auto | off | none
         web:bool=True,                    # let the web tools reach the network
         read_outside:bool=False,          # let reads name any path; writes stay inside
         cfg:str='~/.config/ramabana',     # config dir for skills, extensions and history
         resume:str='',                    # saved session id/prefix, or 'latest'
         attach:str=''):                   # a live dhrishti session by name or base URL
    "Start `ramabana pyrepl` with the normal Ramabana model and project options."
    roots = [item.strip() for item in str(root).split(',') if item.strip()]
    config = Path(cfg).expanduser() if cfg else None
    try: return asyncio.run(amain(roots, model, approve, web, read_outside, config, resume, attach))
    except KeyboardInterrupt: return None

In [ ]:
#| hide
# The import inside `main` is what lets this be swapped; a module-level one would bind the real
# entry point at import time and this test would start a kernel.
import ramabana.cli as _cli, ramabana.pyrepl as _pyrepl
_seen = {}
_real, _pyrepl.main = _pyrepl.main, lambda **kw: _seen.update(kw)
try: _cli.main('pyrepl', root='a,b', model='gpt-mini', web=False)
finally: _pyrepl.main = _real
test_eq((_seen['root'], _seen['model'], _seen['web']), ('a,b', 'gpt-mini', False))
assert 'vault' not in _seen        # pyrepl has no vault-backed host, so the flag is not offered

### Colour

`rich` and `pygments` are already here for the transcript, so highlighting the input line and
the code cells costs no dependency. It is applied to the *rendering* only: `log_cell`, the
session notebook and what `/copy` yields all keep the plain source, because a highlighted
string is not code you can paste.

In [ ]:
#| export
def hl(src):
    """`src` as highlighted `Text`, or plain `Text` if pygments cannot lex it.

    Never raises, and never returns anything whose `.plain` differs from `src`: the transcript
    copies out of `.plain`, so a highlighter that rewrote its input would hand back code that
    does not run.
    """
    src = str(src)
    if not src: return Text('')
    try:
        from rich.syntax import Syntax
        # `highlight` appends a newline -- it is built for whole files. Trimming it back is the
        # difference between a coloured prompt and a prompt one row taller than its line.
        out = Syntax(src, 'python', theme='gruvbox-dark').highlight(src)
        while out.plain.endswith('\n'): out.right_crop(1)
        return out if out.plain == src else Text(src)
    except Exception: return Text(src)

In [ ]:
t = hl('x = 1  # note')
test_eq(t.plain, 'x = 1  # note')            # the text is the text
assert t.spans                                # ...and it carries styles
test_eq(hl('').plain, '')

## The surface

The ordinary Ramabana terminal with one addition: a mode. Agent mode is unchanged — the same
blocks, folding, approvals, slash commands and status bar — and python mode changes what a
*line means*, not what the terminal can do. Python is the default, because this is a REPL
with an agent in it rather than an agent with a REPL in it.

In [ ]:
#| export
PY_HELP = """pyrepl  /agent ask Ramabana · /python run Python · tab complete · ctrl-c interrupt/stop
/promote <name> adopt an agent variable
""" + HELP

async def run_code(ui, code):
    "Run one user-owned Python cell, streaming outputs into Teleprint blocks."
    result = None
    try:
        result = await ui.kernel.execute(code, on_output=ui.on_output)
        if not result.ok and not result.outputs: ui.say(Text(result.error or 'execution failed'), 'error')
    finally:
        if result is not None: ui.agent.host.log_cell(code, result.outputs)
        ui.turn = None
        ui.paint()

async def run_agent(ui, prompt):
    """Stream one model turn and interleave it with Dhrishti's Python tool cells.

    Attachments are taken here, at the start, the way `run_turn` takes them in the base `Ui`:
    the prompt that named them is then the only one that carries them, however the turn goes.
    """
    loop, chunks, events = asyncio.get_running_loop(), [], asyncio.Queue()
    atts, ui.attachments = list(ui.attachments), []
    ask, media = prompt + media_note(atts), media_parts(atts)
    ui.agent.host.log_cell('**user**\n\n' + prompt, cell_type='markdown')
    def pump():
        try:
            for chunk in ui.agent.stream_with(ask, image=media or None):
                loop.call_soon_threadsafe(events.put_nowait, chunk)
        except Exception as exc: loop.call_soon_threadsafe(events.put_nowait, agent_err(exc))
        finally:
            loop.call_soon_threadsafe(events.put_nowait, None)
    threading.Thread(target=pump, daemon=True).start()
    block = None
    try:
        while (chunk := await events.get()) is not None:
            chunks.append(chunk); block = ui.stream(block, chunk)
    finally:
        text = ''.join(chunks)
        if text: ui.agent.host.log_cell('**assistant**\n\n' + text, cell_type='markdown')
        ui.turn = None
        for problem in ui.agent.problems: ui.say(Text(problem), 'error')
        ui.agent.clear_problems(); ui.paint()
    return block

#| export
def _syntax_note(src):
    "What the compile that rejected `src` objected to, in one line."
    return _judge(src)[1]

class PyreplUi(Ui):
    "The ordinary Ramabana terminal with an additional user-owned Python mode."
    #: The continuation prompt: a suite mid-buffer is neither the python nor the agent label,
    #: and using either one here would say the wrong thing about what enter does next. Nine
    #: cells wide, like both labels, so a body's indentation reads at its real depth.
    CONT = '...      '

    def __init__(self, comp, agent, kernel, loop=None):
        self.kernel, self.mode, self.matches = kernel, 'python', None
        super().__init__(comp, agent, loop)

    def status(self):
        out = super().status()
        out.append(f'  · {self.mode}', style=GRUVBOX['aqua'] if self.mode == 'python' else GRUVBOX['blue'])
        return out

    def _label(self):
        "The label ahead of the buffer's first line."
        return 'python › ' if self.mode == 'python' else 'agent  › '

    def _prefixed(self, text):
        """`text` with the mode label on its first line and `CONT` on every line after it.

        The one place either prefix is decided: `prompt` renders it over the whole buffer and
        `tail` over the buffer up to the cursor, so the two cannot disagree about what precedes
        a column. They would have to agree by inspection otherwise, and a continuation row drawn
        with `CONT` but measured without it puts the cursor in the wrong place silently.
        """
        if self.ask is not None: return self.ASKING + text
        return self._label() + text.replace('\n', '\n' + self.CONT)

    def prompt(self):
        if self.ask is not None: return super().prompt()
        color = GRUVBOX['aqua'] if self.mode == 'python' else GRUVBOX['blue']
        label = self._label()
        # Colour only the code itself: splitting the highlighted buffer on its own newlines and
        # rejoining with `CONT` keeps every span's start and end where `hl` put them, so the
        # plain text still matches `_prefixed` exactly -- `tail`'s cursor math (which prefixes
        # the plain buffer text directly, never through `hl`) depends on that agreement.
        # `allow_blank=True` matters here: a buffer awaiting its closing blank line ends with
        # `\n`, and without it `split` drops that trailing empty line -- `str.replace` never
        # would -- which silently ate the very newline `CONT` was meant to follow.
        if self.mode == 'python':
            body = Text('\n' + self.CONT, style=GRUVBOX['fg0']).join(
                hl(self.buf.text).split('\n', allow_blank=True))
        else:
            body = Text(self._prefixed(self.buf.text)[len(label):], style=GRUVBOX['fg0'])
        return Text(label, style=f'bold {color}') + body

    def tail(self):
        rows = [self.status()]
        if self.hint: rows.append(Text(' ' + self.hint, style=GRUVBOX['gray']))
        chips = self.attach_row()
        if chips is not None: rows.append(chips)
        if self.matches: rows.append(Text('  '.join(self.matches[:8]), style=GRUVBOX['gray']))
        rows.append(self.prompt())
        # `render_lines` already does the newlines and the wrapping; the only thing left to get
        # right is prefixing what precedes the cursor exactly as `prompt` prefixed it.
        before = Text(self._prefixed(self.buf.text[:self.buf.cursor]))
        rendered = self.comp.console.render_lines(before, pad=False)
        cursor = (len(rows) - 1, len(rendered) - 1, sum(span.cell_length for span in rendered[-1]))
        return rows, cursor

    async def complete_python(self):
        "Kernel completion for the buffer, with the displayed list annotated from the namespace."
        # The identity a completion is computed *for*: buffer text and cursor together, since
        # either changing means the candidates no longer describe what is on screen. Checked
        # after every `await` below -- the kernel round trip and the off-thread `describe` lookup
        # are both places ordinary typing can run underneath this coroutine: `Compositor` only
        # awaits a key handler that *returns* a coroutine, and a plain character key does not, so
        # nothing gates the buffer while either await is outstanding. Two checks, not one: the
        # two awaits are independent gaps -- the buffer can move on during either without moving
        # on during the other -- so a single check before both assignments would miss whichever
        # gap it wasn't next to.
        identity = (self.buf.text, self.buf.cursor)
        def stale(): return (self.buf.text, self.buf.cursor) != identity
        try:
            matches, start = await self.kernel.complete(self.buf.text, self.buf.cursor)
            if not matches or stale(): return
            common = os.path.commonprefix(matches)
            if len(matches) == 1 or len(common) > self.buf.cursor - start:
                self.buf.text = self.buf.text[:start] + common + self.buf.text[self.buf.cursor:]
                self.buf.cursor = start + len(common)
                identity = (self.buf.text, self.buf.cursor)   # our own edit, not the user moving on
            if len(matches) <= 1:
                self.matches = None
                return
            # Bare names paint first, before the namespace is ever asked about them: `describe`
            # is a synchronous HTTP call, and running it inline here -- even with its own 2s
            # timeout -- would freeze the whole event loop for up to 2s on every tab press, not
            # just the completion list: the spinner, the status bar and every other keystroke
            # queue up behind it. `to_thread` moves the blocking call off the loop, and the
            # names go up immediately so "best effort" means what it says: a list that is never
            # empty and never waits on the network to appear at all.
            self.matches = list(matches)
            self.paint()
            described = await asyncio.to_thread(getattr(self.agent.host, 'describe', dict))
            if stale(): return   # the buffer moved on while `describe` was in flight; drop it
            self.matches = annotate(matches, described)
        finally:
            self.turn = None
            self.paint()

    def submit(self):
        """Handle the typed line. Mode switches and `/help` are this surface's own; every other
        slash command -- `/attach`, `/detach`, `/paste`, `/copy`, and whatever the agent
        implements -- is the same command in both modes, so the base `Ui` handles it. A plain
        line means whichever mode owns it: a Python line in python mode, or a turn (with
        whatever `@path` names and whatever is already attached) in agent mode.
        """
        line = self.buf.text.strip()
        self.matches = None
        if not line:
            self.buf.clear()
            return None
        if line in ('/agent', '/a'):
            self.buf.clear()
            self.mode = 'agent'; self.say(Text('agent mode'), 'note', fold=None)
            return None
        if line in ('/python', '/py'):
            self.buf.clear()
            self.mode = 'python'; self.say(Text('python mode'), 'note', fold=None)
            return None
        if line in ('/help', '/?'):
            self.buf.clear()
            self.say(Text(PY_HELP), 'note', fold=None)
            return None
        if line.split()[0] in ('/promote', '/adopt') and len(line.split()) == 2:
            base = self.kernel.base if self.kernel else ''
            self.buf.clear()
            self.say(Text(promote(base, line.split()[1])), 'note', fold=None)
            return None
        if line.startswith('/'): return super().submit()
        if self.mode == 'python':
            self.buf.clear()
            self.say(hl(line), 'user', source=line)
            return run_code(self, line)
        # Agent mode: a plain line is a turn, and `@path` inside it attaches like everywhere
        # else -- a python line stays literal text, so this parsing never runs there.
        self.buf.clear()
        got = [self.attach(p) for p in attach_refs(line)]
        if got: self.note('\n'.join(got))
        self.say(Text(line), 'user')
        return run_agent(self, line)

    def on_output(self, output):
        kind = output.get('output_type')
        if kind == 'stream':
            text = _text(output.get('text')).rstrip('\n')
            if text: self.say(Text(text), 'note')
        elif kind in ('execute_result', 'display_data'):
            data = output.get('data') or {}
            if 'text/plain' in data: self.say(Text(_text(data['text/plain'])), 'reply')
            elif 'text/markdown' in data: self.say(Text(_text(data['text/markdown'])), 'reply')
            elif data: self.say(Text('display: ' + ', '.join(data)), 'note')
        elif kind == 'error':
            body = '\n'.join(output.get('traceback') or [])
            self.say(Text.from_ansi(body or f"{output.get('ename')}: {output.get('evalue')}"), 'error')

    def on_key(self, key):
        # Tab completes in python mode only, and only when there is something to complete and
        # nothing already running -- an empty buffer or an in-flight cell makes tab a no-op
        # rather than a request the kernel would answer with nothing useful.
        if (key.name == 'tab' and self.mode == 'python' and self.ask is None
                and self.turn is None and self.buf.text):
            return self.complete_python()
        # A typed suite grows the buffer instead of submitting it: `codeop` is asked once per
        # keystroke rather than the kernel, so an unfinished `for` costs nothing to try. This
        # sits before the ctrl-c branch and is guarded the same way the transcript-priority
        # patch on `Ui.on_key` guards its own use of `enter` -- not while an approval, a menu
        # or the transcript view already owns the key.
        # `startswith('/')` is why this is not just a python-mode check: a slash line is a
        # command, not code, and `/agent` is invalid Python -- so this branch reported "invalid
        # syntax" and returned before `submit` ever saw it, which left no way out of python mode
        # at all. Commands belong to `submit`; only code is compiled here.
        if (key.name == 'enter' and self.mode == 'python' and self.ask is None
                and self.menu is None and not self.transcript.active
                and not self.buf.text.lstrip().startswith('/')):
            src = self.buf.text
            # A blank line submits what is pending, which is the only way to close a suite and
            # what fingers already do: the trailing newline is what makes `code_state` say
            # complete. `endswith('\n\n')` is the way out of a buffer that never will -- an
            # empty suite body, an unclosed bracket -- which would otherwise grow forever; a
            # second blank line submits it and lets the kernel name the error. An empty buffer
            # falls through untouched, so a bare enter still just clears the line.
            if src.strip() and not src.endswith('\n\n'):
                state = code_state(src)
                if state == 'incomplete':
                    self.buf.insert('\n'); return self.paint()
                if state == 'invalid':
                    # Said here rather than by the kernel: a syntax error costs no execution,
                    # and a prompt that answers instantly is the difference this makes.
                    self.say(Text(f'incomplete or invalid: {_syntax_note(src)}'), 'error', fold=None)
                    return self.paint()
        # ctrl-c means "stop what is running", and in python mode what is running is a cell.
        if key.name == 'ctrl+c' and self.turn is not None and self.mode == 'python':
            self.comp.spawn(self.kernel.interrupt(), name='interrupt')
            self.buf.clear(); return self.paint()
        self.matches = None
        return super().on_key(key)

In [ ]:
from ramabana.testing import fake_agent
from teleprint.compositor import Compositor
from teleprint.keys import Key
from teleprint.testing import EmuTty        # not `pyghostty.EmuTty`; see nbs/05_cli.ipynb

# Tall enough that the verbose /help block in Step 5 stays on the visible screen: `tty.term.text()`
# is the active area only, no scrollback, and `PY_HELP` plus the shared `HELP` wrap past 16 rows.
tty = EmuTty(80, 40)
comp = Compositor(tty)
comp._register_signals = lambda: None    # nbdev runs this async cell on a worker thread
await comp.start()
agent, be = fake_agent(host=host, replies=['`owner` is a dict with one key.'])
ui = PyreplUi(comp, agent, kernel)
comp.on_key = ui.on_key
ui.paint()
test_eq(ui.mode, 'python')                       # python owns the line by default
assert 'python' in ui.status().plain

In [ ]:
# Switching is a slash command, so it costs no key and cannot be typed by accident.
ui.buf.insert('/agent'); ui.submit()
test_eq(ui.mode, 'agent')
assert 'agent' in ui.prompt().plain
ui.buf.insert('/py'); ui.submit()
test_eq(ui.mode, 'python')

# Every other slash command still reaches the agent: agent mode is not a different program.
ui.buf.insert('/help'); ui.submit()
assert 'pyrepl' in tty.term.text()

In [ ]:
# Each output kind lands in the block that means it, so a traceback never reads as a reply.
ui.on_output({'output_type': 'stream', 'name': 'stdout', 'text': 'printed\n'})
ui.on_output({'output_type': 'execute_result', 'data': {'text/plain': '42'}})
ui.on_output({'output_type': 'error', 'ename': 'ValueError', 'evalue': 'bad', 'traceback': []})
kinds = [b.tag for b in comp.blocks.values()][-3:]
test_eq(kinds, ['note', 'reply', 'error'])

In [ ]:
# A python line executes in the owner's kernel and is logged; an agent line asks the model.
await run_code(ui, 'from_the_prompt = 99')
r = await kernel.execute('from_the_prompt')
test_eq(output_text(r.outputs), '99')
ui.mode = 'agent'
await run_agent(ui, 'what is in owner?')
assert be.sent, 'the turn reached the backend'
test_eq([c.cell_type for c in read_nb(host.agent_log).cells[-2:]], ['markdown', 'markdown'])
ui.mode = 'python'

In [ ]:
# Agent mode keeps the whole Ramabana surface, attachments included: `/attach` and `@path`
# both work, the chip shows what the next turn carries, and the picture itself -- not a
# description of it -- reaches the backend as a content part.
pics = Path(tempfile.mkdtemp())
(pics / 'shot.png').write_bytes(b'\x89PNG\r\n\x1a\n' + b'0' * 64)
shot = pics / 'shot.png'

ui.mode = 'agent'
ui.buf.insert(f'/attach {shot}'); ui.submit()
assert 'shot.png' in ui.attach_row().plain
ui.buf.insert('what is in the picture?')
await ui.submit()
assert isinstance(be.sent[-1], list) and shot.read_bytes() in be.sent[-1]
test_eq(ui.attachments, [])                      # taken at the start of the turn, not left behind
ui.mode = 'python'

In [ ]:
# `@` inside a python line is Python's own syntax, not a file reference: `attach_refs` lives
# only in the agent-mode branch of `submit`, so a python line never reaches it. This would
# break if that call moved above the mode split, or `submit` were "simplified" into an
# unconditional delegation to the base `Ui`.
leak = pics / '@leak.png'
leak.write_bytes(b'\x89PNG\r\n\x1a\n' + b'0' * 64)   # a real file, named to look like a reference

ui.mode = 'python'
ui.buf.insert("py_matmul = type('M', (), {'__matmul__': lambda self, o: 42})() @ 1")
await ui.submit()
r = await kernel.execute('py_matmul')
test_eq(output_text(r.outputs), '42')          # `@` ran as matrix multiplication, not a path
test_eq(ui.attachments, [])

ui.buf.insert(f'py_leak = 5  # not an attachment: @{leak}')
await ui.submit()
r = await kernel.execute('py_leak')
test_eq(output_text(r.outputs), '5')           # the line ran, `@leak.png` and all
test_eq(ui.attachments, [])                    # a python line never calls `attach_refs`

### Typing more than one line

`Ui` submits on `enter`, which is right for a sentence and wrong for a suite: `for i in
range(3):` is not a program yet. `codeop.compile_command` is the standard answer and tells the
three cases apart — this compiles, this is unfinished, this will never compile — so an
unfinished line grows the buffer and a broken one is reported at the prompt without a round
trip to the kernel.

An empty line submits what is pending, which is what every Python prompt does and what
fingers expect.

In [ ]:
#| export
def _compile_state(src, symbol):
    "`compile_command` as a (state, message) pair rather than three shapes of answer."
    try:
        code = codeop.compile_command(str(src), '<pyrepl>', symbol)
        return ('complete' if code is not None else 'incomplete'), ''
    except SyntaxError as e: return 'invalid', f'{e.msg} (line {e.lineno})'
    except Exception as e: return 'complete', agent_err(e)   # not ours to judge; let the kernel say

def _judge(src):
    """The prompt's verdict on `src`, and what objected when it was rejected.

    `'single'` is the primary judgement because it asks the question a REPL asks: a suite is
    unfinished until its blank line, where `'exec'` calls it finished one line after the colon
    and would submit a `for` with no body typed for it yet.

    But `'single'` also rejects two valid statements -- "multiple statements found" -- and a
    paste arrives here as a whole buffer rather than a line at a time. A real terminal never
    meets that case because readline compiles each pasted line as it streams, so `'single'`
    never sees more than one statement; this surface asks once, about everything. So a
    `'single'` rejection is checked against `'exec'` before it is believed, and `'exec'`'s
    verdict stands when it has one. Code that is genuinely broken is rejected by both, and then
    it is `'exec'` that names the real reason rather than counting the statements.
    """
    state, note = _compile_state(src, 'single')
    if state != 'invalid': return state, note
    return _compile_state(src, 'exec')

def code_state(src):
    "Whether `src` is a finished statement, an unfinished one, or one that cannot compile."
    return _judge(src)[0]

In [ ]:
test_eq(code_state('x = 1'), 'complete')
test_eq(code_state('for i in range(3):'), 'incomplete')
test_eq(code_state('x = ('), 'incomplete')
test_eq(code_state('x = )'), 'invalid')
test_eq(code_state(''), 'complete')
# A suite needs its blank line before it compiles, exactly as at a real prompt.
test_eq(code_state('for i in range(3):\n    print(i)'), 'incomplete')
test_eq(code_state('for i in range(3):\n    print(i)\n'), 'complete')

# Two statements are one paste, and `'single'` on its own calls them invalid; `'exec'` is asked
# before that verdict is believed, so a real snippet is not reported as broken.
test_eq(code_state('x = 1\ny = 2\n'), 'complete')
test_eq(code_state('import os\nprint(os.getcwd())'), 'complete')
test_eq(code_state('import os\nfor i in range(2):\n    print(i)'), 'complete')

# The fallback does not swallow anything: broken is broken under both, and the message the user
# sees is the real objection rather than a count of statements.
test_eq(code_state('x = 1\ny = )'), 'invalid')
assert 'unmatched' in _syntax_note('x = 1\ny = )')
assert 'multiple statements' not in _syntax_note('x = 1\ny = )')
test_eq(code_state('x = 1\ny = ('), 'incomplete')     # unfinished, not invalid: it grows

In [ ]:
ui.buf.clear()
ui.buf.insert('for i in range(3):')
ui.on_key(Key('enter'))
test_eq(ui.buf.text, 'for i in range(3):\n')       # grew, rather than being submitted
assert ui.turn is None
assert ui.CONT.strip() in tty.term.text()          # ...and it says so

ui.buf.insert('    print(i)')
ui.on_key(Key('enter'))                            # a suite still needs its blank line
test_eq(ui.buf.text, 'for i in range(3):\n    print(i)\n')

In [ ]:
# `prompt()` and `tail()` prefix through one helper, so the cursor cannot drift: a continuation
# row drawn with `CONT` but measured without it lands in the wrong column while nothing raises.
# Checked against `CONT` plus what was typed on that row, rather than by eye.
ui.buf.clear()
ui.buf.insert('for i in range(3):')
ui.on_key(Key('enter'))                            # two rows now, with the cursor on the second
ui.buf.insert('    print(i)')
rows, (row, line_no, col) = ui.tail()
test_eq(row, len(rows) - 1)                        # the prompt is the last tail row...
test_eq(line_no, 1)                                # ...the cursor is on its second visual line...
test_eq(col, Text(ui.CONT + '    print(i)').cell_len)   # ...at the column that prefix implies
assert ui.CONT in ui.prompt().plain                # and the row really is drawn with it

ui.on_key(Key('enter'))                            # back where the previous cell left off
test_eq(ui.buf.text, 'for i in range(3):\n    print(i)\n')

In [ ]:
ui.on_key(Key('enter'))                       # the blank line closes it
test_eq(ui.buf.text, '')
ui.buf.clear()

# Invalid is reported now, not after a round trip.
ui.buf.insert('x = )')
ui.on_key(Key('enter'))
assert 'invalid' in tty.term.text() and ui.turn is None
ui.buf.clear()

# Agent mode is untouched: a question that happens to look like a suite is still a question.
ui.mode = 'agent'
ui.buf.insert('for the record:')
ui.on_key(Key('enter'))
test_eq(ui.buf.text, '')
ui.mode = 'python'

In [ ]:
# A paste arrives as one buffer rather than a line at a time, so a two-statement snippet has to
# submit as it stands. The kernel is the only witness that both statements ran: a fallback that
# compiled the block and then executed only its head would look identical from up here.
ui.buf.clear()
ui.paste('pasted_a = 3\npasted_b = pasted_a * 2')
test_eq(code_state(ui.buf.text), 'complete')
out = ui.on_key(Key('enter'))
test_eq(ui.buf.text, '')                       # submitted, not reported as invalid
if out is not None: await out
r = await kernel.execute('pasted_b')
test_eq(output_text(r.outputs), '6')           # the second statement ran too, not just the first

In [ ]:
# Coloured on screen, plain on the way out: `hl` renders the prompt and the transcript block,
# but `/copy` and `block_text` both read `source`, which was never touched by it.
ui.buf.clear(); ui.buf.insert('df2 = df.copy()')
assert ui.prompt().spans                                   # coloured on screen...
blk = ui.say(hl('df2 = df.copy()'), 'user', source='df2 = df.copy()')
test_eq(ui.transcript.block_text(blk), 'df2 = df.copy()')  # ...and plain on the way out
ui.buf.clear()

# The cursor test from the earlier "typing more than one line" section still has to land on the
# same column with colour turned on: `prompt().plain` is unchanged by `hl`, and `tail`'s cursor
# math never calls `hl` at all, so this is the regression `hl` could have introduced and didn't.
ui.buf.insert('for i in range(3):')
ui.on_key(Key('enter'))
ui.buf.insert('    print(i)')
rows, (row, line_no, col) = ui.tail()
test_eq(row, len(rows) - 1)
test_eq(line_no, 1)
test_eq(col, Text(ui.CONT + '    print(i)').cell_len)
assert ui.CONT in ui.prompt().plain
assert ui.prompt().spans                                    # this time, coloured too
ui.buf.clear(); ui.on_key(Key('enter'))  # discard: an empty line submits nothing

In [ ]:
# `run_code` never sees a highlighted line -- `submit` colours the transcript block but hands
# `run_code` (and so `log_cell`) the plain `line` it started with -- so the session notebook
# gets code that runs, not a rendering of it.
await run_code(ui, 'colour_check = 1  # a comment, to give `hl` spans to find')
test_eq(read_nb(host.agent_log).cells[-1].source, 'colour_check = 1  # a comment, to give `hl` spans to find')

### Getting back out of python mode

A slash line is a command, not code. `/agent` is invalid Python, so compiling the buffer before
looking at it reported "invalid syntax" and returned -- and since the check ran on every `enter`,
there was no way to leave python mode at all. Commands go to `submit`; only code is compiled.

In [ ]:
def _line(u, s):
    for ch in s: u.on_key(Key(ch, char=ch))
    return u.on_key(Key('enter'))

test_eq(code_state('/agent'), 'invalid')     # it really is not Python, which was the trap
_line(ui, '/agent')
test_eq((ui.mode, ui.buf.text), ('agent', ''))
_line(ui, '/python')
test_eq((ui.mode, ui.buf.text), ('python', ''))
_line(ui, '/help')                            # and the surface's own commands still answer
test_eq((ui.mode, ui.buf.text), ('python', ''))
assert 'pyrepl' in tty.term.text()

# ...while a real suite is still compiled and still grows instead of submitting.
_line(ui, 'for i in range(3):')
test_eq(ui.buf.text, 'for i in range(3):\n')
ui.buf.clear()

The session does not hold the mouse. `cli.amain` makes the same choice: the main screen belongs
to the terminal, so selecting and copying there work as they do in any scrollback, and
`enter_transcript` borrows the mouse only while the browsing view is up. Holding it for the whole
session stole that selection *and* streamed every mouse move in as input, redrawing the transcript
once per event.

In [ ]:
import inspect
src = inspect.getsource(amain)
assert '?2004h' in src                        # bracketed paste, so a paste arrives as one event
assert '?1000' not in src.split('finally')[0]  # ...and no mouse reporting on the way in

### Tab, against the live kernel

One match completes the buffer in place; several list, with the descriptions coming from
whatever the session is actually holding.

In [ ]:
await kernel.execute('completing = {"a": 1}')
ui.buf.clear(); ui.buf.insert('complet')
ui.buf.cursor = len(ui.buf.text)
await ui.complete_python()
test_eq(ui.buf.text, 'completing')             # one match completes in place
ui.buf.clear()

# Several matches list, and the ones that name live values say what they are. Asserted against
# names this cell created, so the claim is exact rather than "something was returned".
await kernel.execute('twovals_a = 1\ntwovals_b = 2')
ui.buf.clear(); ui.buf.insert('twovals_'); ui.buf.cursor = len('twovals_')
await ui.complete_python()
assert ui.matches is not None and len(ui.matches) == 2, ui.matches
test_eq(sorted(m.split(' → ')[0] for m in ui.matches), ['twovals_a', 'twovals_b'])
assert all('→' in m for m in ui.matches), ui.matches      # both are live, so both are described
ui.buf.clear()

In [ ]:
# `describe` is synchronous HTTP; `complete_python` must not call it on the loop thread, or a
# slow or dead dhrishti host freezes the whole event loop -- spinner, status bar and keystrokes
# included -- for up to its 2s timeout on every tab press. Checked directly: the thread
# `describe` actually runs on is recorded and compared against the thread driving this test
# (which is the loop thread, since `await` on it never leaves that thread).
loop_thread = threading.current_thread()
seen_thread = {}
_real_describe = host.describe
def _watched_describe():
    seen_thread['t'] = threading.current_thread()
    return _real_describe()
host.describe = _watched_describe
try:
    await kernel.execute('threadcheck_a = 1\nthreadcheck_b = 2')
    ui.buf.clear(); ui.buf.insert('threadcheck_'); ui.buf.cursor = len('threadcheck_')
    await ui.complete_python()
finally:
    host.describe = _real_describe
assert seen_thread.get('t') is not None and seen_thread['t'] is not loop_thread
assert all('→' in m for m in ui.matches), ui.matches   # the off-thread lookup still landed
ui.buf.clear()

In [ ]:
# Moving `describe` off-thread opened a race: ordinary typing is never gated by `self.turn`
# (`Compositor` only awaits a handler that *returns* a coroutine, and a character key does not),
# so the buffer can move on while a `describe` lookup is still in flight. Made deterministic
# with a `describe` that blocks on an `Event` until released, so the user's next keystroke can
# land at a guaranteed point *inside* the await -- a real race would only sometimes catch this.
release = threading.Event()
def _blocked_describe():
    release.wait(5)
    return {'twovals_a': 'int', 'twovals_b': 'int'}
_real_describe = host.describe
host.describe = _blocked_describe
ui.matches = None   # clear whatever a previous test left behind, so the poll below is honest
try:
    await kernel.execute('twovals_a = 1\ntwovals_b = 2')
    ui.buf.clear(); ui.buf.insert('twovals_'); ui.buf.cursor = len('twovals_')
    task = asyncio.ensure_future(ui.complete_python())
    # Wait for the bare-names paint: `self.matches` holds the raw list, no `→` yet, which is
    # exactly the point at which `describe` has been handed to its thread and is blocked.
    for _ in range(500):
        if ui.matches: break
        await asyncio.sleep(0.01)
    assert ui.matches and not any('→' in m for m in ui.matches)   # bare, not yet annotated
    ui.on_key(Key('x', char='x'))                    # the user keeps typing while describe blocks
    test_eq(ui.buf.text, 'twovals_x')
    test_eq(ui.matches, None)                        # typing cleared it immediately, synchronously
    release.set()                                    # let the blocked lookup return
    await task
finally:
    host.describe = _real_describe
    release.set()
test_eq(ui.matches, None)          # the stale annotate() never landed over what typing cleared
test_eq(ui.buf.text, 'twovals_x')  # and the buffer kept the user's edit, untouched by the reply
ui.buf.clear()

## `/promote`

Promotion is the owner's act, so it is a command and never a tool: the model can ask for it in
words, and the person at the prompt is the one who does it. Own-kernel mode only -- attach mode
has no owner token to read, so the session belongs to whoever started it, and they promote from
their own surface.

In [ ]:
#| export
def promote(base, name):
    """Adopt an agent-layer variable into the owner namespace. Own-kernel mode only.

    The one call here that needs the owner token, and the reason it is a slash command rather
    than a tool: the token exists so that adopting an agent's work is a decision a person makes.
    In attach mode there is no token to read -- the session belongs to whoever started it, and
    they promote from their own surface.
    """
    if not base: return 'not attached to a session'
    from dhrishti.serving import owner_token
    port = int(str(base).rsplit(':', 1)[-1])
    if (tok := owner_token(port)) is None: return f'no owner token for {base}; this session is not ours to change'
    try: result = _api(base, '/api/promote', {'accessor': json.dumps([str(name)]), 'token': tok})
    except Exception as exc: return agent_err(exc)
    return str(result.get('error') or f'promoted {name}')

In [ ]:
# Promotion is the owner's act, so it is a command and never a tool: the model can ask for it
# in words, and the person at the prompt is the one who does it.
assert 'not attached' in promote('', 'x') or 'no session' in promote('', 'x')

In [ ]:
# `/promote` reaches the live ui the same way `/agent` and `/python` do: through `submit`,
# which clears the buffer and answers with a note rather than falling through to the agent.
ui.buf.insert('/promote x')
test_eq(ui.submit(), None)
test_eq(ui.buf.text, '')

### Attaching to the live session

The kernel this notebook's own Task 2 cell started is itself a live dhrishti session, so
`find_session` can locate it by the name the registry gave it -- proof that attach works
against a real server, not a fake one. The name is `'<project>-pyrepl-<pid>'` (see
`Kernel._bootstrap`), unique even next to a stale same-named-in-spirit leftover from an
earlier, unrelated run: the pid is what makes this deterministic without depending on the
machine's registry being otherwise empty.

In [ ]:
# The session this notebook's own kernel started is a live one, so attach can find it by name.
# Deterministic even with other dhrishti sessions alive on this machine (other repos, other
# terminals, a stale leftover process): `Kernel._bootstrap` names each session
# '<project>-pyrepl-<pid>', so no other live entry can share this one's exact name.
from dhrishti.serving import active
own = next((e for e in active() if e.get('base') == kernel.base), None)
assert own, 'the kernel from Task 2 is serving'
test_eq(find_session(own['name']), kernel.base)

# An attached host over the discovered base is an ordinary host -- no kernel involved.
attached = DhrishtiHost(['.'], find_session(own['name']), web=False, index=False)
test_eq(attached.kernel_kind, 'ipykernel')
assert 'owner' in attached.list_vars()

In [ ]:
await kernel.shutdown()
test_eq(kernel.alive, False)